
# PAWS-X (Spanish) Paraphrase Detection — **Student Version**

**Goal:** Fine-tune a multilingual Transformer (**XLM-RoBERTa**) on the **PAWS-X (es)** dataset to detect whether two sentences are paraphrases (binary classification).

## Learning objectives
- Load and inspect a sentence-pair classification dataset (PAWS-X, Spanish split).
- Tokenize **pairs** of sentences with a Transformer tokenizer (XLM-R).
- Build a sequence classification head for binary labels.
- Train and evaluate using Hugging Face `Trainer`.
- Run inference on custom examples.

> 🧭 **Instructions:** Many cells contain `TODO` markers. Replace them with working code. Keep your code clean and add brief comments.



## 1) (Optional) Install/upgrade libraries
**What to do:** If you're in a fresh environment, install required libraries.  
If your environment is managed (e.g., provided by your instructor), you may skip this cell.


In [ ]:

# TODO (optional): Uncomment and run if needed.
# !pip -q install -U transformers datasets evaluate accelerate sentencepiece
# If you use cuDF/RAPIDS and face a pyarrow conflict, you might need:
# !pip -q install "pyarrow<20"



## 2) Imports and experiment setup
**What to do:** Import libraries, print versions, pick device (GPU/CPU), and set a random seed for reproducibility.


In [ ]:

# TODO: import the required libraries.
# Hints:
# - datasets.load_dataset
# - transformers: AutoTokenizer, AutoModelForSequenceClassification,
#                 DataCollatorWithPadding, TrainingArguments, Trainer
# - evaluate
# - torch, numpy, random, inspect (optional)

# TODO: print library versions (transformers, datasets) and the selected device.
# TODO: set a global random seed (e.g., 42).
raise NotImplementedError("Implement imports, version prints, device selection, and seeding.")



## 3) Load PAWS-X (Spanish)
**What to do:** Load the **Spanish** configuration of PAWS-X. Inspect the splits and a sample example.

**Expected columns:** `sentence1`, `sentence2`, `label` (0 = not-paraphrase, 1 = paraphrase).  
**Tip:** Keep a mapping `id2label = {0: "not-paraphrase", 1: "paraphrase"}` and its inverse.


In [ ]:

# TODO: load the dataset -> load_dataset("paws-x", "es")
# TODO: print the dataset object and one training sample
# TODO: define id2label and label2id dicts
raise NotImplementedError("Load PAWS-X (es) and define label maps.")



## 4) Tokenizer and preprocessing for **sentence pairs**
**What to do:**
1. Initialize the tokenizer for `xlm-roberta-base` (use fast tokenizer).
2. Write a `preprocess(batch)` function that tokenizes `(sentence1, sentence2)` with truncation & padding.
3. Map the function over `train/validation/test`. Remove non-tensor columns (`id`/`idx`, `sentence1`, `sentence2`) **only if present** to avoid errors.
4. Rename `label` → `labels` and set PyTorch format.

**Hints:**
- Use `padding="max_length"` and `max_length=256` initially (you can change later).
- A helper like `cols_present = [c for c in to_maybe_remove if c in dataset[split].column_names]` can be handy.


In [ ]:

# TODO: tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-base", use_fast=True)
# TODO: define preprocess(batch) that tokenizes (sentence1, sentence2)
# TODO: map over splits and remove only existing columns among ["idx","id","sentence1","sentence2"]
# TODO: rename "label" -> "labels" and set format to torch
# Finally: assign to train_ds, eval_ds, test_ds
raise NotImplementedError("Implement tokenizer + preprocessing + dataset mapping.")



## 5) Model: XLM-R for binary classification
**What to do:** Load `AutoModelForSequenceClassification` with `num_labels=2` and your `id2label/label2id`. Move it to the selected device.


In [ ]:

# TODO: instantiate the model from "xlm-roberta-base" with 2 labels and your label maps.
# TODO: move model to device.
raise NotImplementedError("Instantiate XLM-R (sequence classification) and send to device.")



## 6) Metrics
**What to do:** Using `evaluate`, load accuracy, precision, recall, and F1. Implement `compute_metrics(eval_pred)` that returns:
- `accuracy`
- `precision_macro`
- `recall_macro`
- `f1_macro`


In [ ]:

# TODO: metric loaders via evaluate.load("accuracy"), etc.
# TODO: define compute_metrics(eval_pred) that softmax/argmax logits to preds and computes the metrics above.
raise NotImplementedError("Define evaluation metrics (accuracy, macro P/R/F1).")



## 7) Training configuration
**What to do:** Build `TrainingArguments` with sensible defaults:
- `output_dir="xlmr-pawsx-es"`
- batch sizes (e.g., 16)
- `num_train_epochs=3`
- `learning_rate=2e-5`, `weight_decay=0.01`
- Logging every ~100 steps

**Important:** Ensure evaluation and save strategies **match** (e.g., both `"epoch"`), **or** use `eval_steps`/`save_steps` (for older Transformers). Avoid setting `load_best_model_at_end=True` unless both strategies exist and match in your environment.


In [ ]:

# TODO: build TrainingArguments with matching eval/save strategies (or steps fallback)
# Optionally add warmup_ratio, report_to="none"
raise NotImplementedError("Create TrainingArguments safely (no strategy mismatch).")



## 8) Trainer and fine-tuning
**What to do:**
- Create a `DataCollatorWithPadding` from your tokenizer.
- Instantiate `Trainer` with model, args, datasets, tokenizer, data_collator, and `compute_metrics`.
- Call `trainer.train()`.


In [ ]:

# TODO: data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
# TODO: instantiate Trainer(...)
# TODO: run trainer.train()
raise NotImplementedError("Instantiate Trainer and fine-tune the model.")



## 9) Evaluation on the test split
**What to do:** Evaluate your best (or final) model on `test_ds` and display the metrics.


In [ ]:

# TODO: test_metrics = trainer.evaluate(test_ds); display the dict
raise NotImplementedError("Evaluate on the test split and show metrics.")



## 10) Inference helper
**What to do:** Implement a function `predict_paraphrase(s1, s2)` that:
- Tokenizes the pair
- Runs the model
- Returns predicted label (`not-paraphrase` / `paraphrase`) and the confidence score

**Tip:** Use `softmax` over logits, then `argmax`.


In [ ]:

# TODO: implement predict_paraphrase(sentence1, sentence2) -> dict with keys:
# 'sentence1', 'sentence2', 'prediction', 'score'
raise NotImplementedError("Write predict_paraphrase().")



## 11) Try at least **five** examples
**What to do:** Print predictions and scores.


In [ ]:

# TODO: define a list of 5 (s1, s2) pairs and loop over them, calling predict_paraphrase

# examples = [
#     # 1) Given example (likely paraphrase)
#     ("El perro persigue al gato por el jardín.",
#      "En el jardín, el perro va detrás del gato."),

#     # 2) Clear paraphrase
#     ("Madrid es la capital de España.",
#      "La capital de España es Madrid."),

#     # 3) Not paraphrase (landed vs took off)
#     ("El avión aterrizó a las 7.",
#      "El avión despegó a las 7."),

#     # 4) Likely paraphrase (preference phrasing)
#     ("Me gusta el café sin azúcar.",
#      "Prefiero el café sin azúcar."),

#     # 5) Not paraphrase (bought vs sold)
#     ("Juan compró un coche rojo.",
#      "Juan vendió un coche rojo."),
# ]

# results = [predict_paraphrase(a, b) for a, b in examples]
# for i, r in enumerate(results, 1):
#     print(f"Example {i}: {r['prediction']} (score={r['score']:.3f})")
#     print("  s1:", r["sentence1"])
#     print("  s2:", r["sentence2"])
#     print("-"*60)

raise NotImplementedError("Run five inference examples.")
